# Import the data from CIQ from the excel sheet

In [1]:
import pandas as pd
import os

cwd = os.getcwd()
cwd

'c:\\Users\\adamf\\PycharmProjects\\FedContracts_AlgoTrading'

In [2]:
filepath = rf"{cwd}\datsets\fundamentals\ciq_eps_revision_raw.csv"

In [9]:
raw = pd.read_csv(filepath, encoding="cp1252")
raw.head()

,SPGTable,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,Entity Name,Entity ID,Exchange,Market Capitalization ($M),CIQ EPS Est - # of Est (actual),EPS_FY+1_Current,EPS_FY+1_1M_ago,EPS_FY+1_3M_ago,Rev_1M,Rev_3M
1,SP_ENTITY_NAME,SP_ENTITY_ID,SP_EXCHANGE,SP_MARKETCAP,SP_EPS_NUM_EST,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,Current,FY+1,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN
4,"10x Genomics, Inc. (NASDAQGS:TXG)",5248867,NASDAQGS,"2,944.69",15,-0.97,-0.49,-0.52,-0.993459213,-0.852054271


In [10]:
raw.columns = raw.iloc[0,:]
raw.drop([0,1,2,3], axis=0, inplace=True)
raw.head()

,Entity Name,Entity ID,Exchange,Market Capitalization ($M),CIQ EPS Est - # of Est (actual),EPS_FY+1_Current,EPS_FY+1_1M_ago,EPS_FY+1_3M_ago,Rev_1M,Rev_3M
4,"10x Genomics, Inc. (NASDAQGS:TXG)",5248867,NASDAQGS,"2,944.69",15,-0.97,-0.49,-0.52,-0.993459213,-0.852054271
5,1st Source Corporation (NASDAQGS:SRCE),100444,NASDAQGS,"1,630.36",3,6.64,6.64,6.48,0,0.025205247
6,3M Company (NYSE:MMM),105135,NYSE,"87,074.04",18,8.66,8.67,8.04,-0.000651787,0.077737289
7,A. O. Smith Corporation (NYSE:AOS),4020605,NYSE,"10,765.87",14,4.02,3.80,3.80,0.058888002,0.057547785
8,"A10 Networks, Inc. (NYSE:ATEN)",4433033,NYSE,"1,377.63",6,1.01,0.88,0.88,0.153206543,0.153206543


In [12]:
# Clean column names
raw.columns = [c.strip().lower().replace(" ", "_").replace("+", "") for c in raw.columns]

# Rename to consistent names
raw = raw.rename(columns={
    "entity_name": "entity_name_raw",
    "entity_id": "iqid",
    "market_capitalization_($m)": "mktcap_usd_m",
    "ciq_eps_est_-_#_of_est_(actual)": "n_est_fy1",
})

# Ensure numeric
num_cols = [
    "mktcap_usd_m", "n_est_fy1",
    "eps_fy1_current", "eps_fy1_1m_ago", "eps_fy1_3m_ago",
    "rev_1m", "rev_3m"
]
# remove thousands separators, then convert safely
raw[num_cols] = (
    raw[num_cols]
    .replace({',': ''}, regex=True)
    .apply(pd.to_numeric, errors='coerce')
    .astype(float)
)


# Drop obvious junk rows
raw = raw.dropna(subset=["iqid", "rev_1m", "rev_3m"])

# Optional: remove cases where denominator was ~0, which can explode revisions
raw = raw[raw["eps_fy1_1m_ago"].abs() > 1e-6]
raw = raw[raw["eps_fy1_3m_ago"].abs() > 1e-6]

# Add as-of date for your pipeline
raw["asof_date"] = pd.Timestamp.today().normalize()

raw.head()

,entity_name_raw,iqid,exchange,mktcap_usd_m,n_est_fy1,eps_fy1_current,eps_fy1_1m_ago,eps_fy1_3m_ago,rev_1m,rev_3m,asof_date
4,"10x Genomics, Inc. (NASDAQGS:TXG)",5248867,NASDAQGS,2944.69,15.0,-0.97,-0.49,-0.52,-0.993459,-0.852054,2026-03-02
5,1st Source Corporation (NASDAQGS:SRCE),100444,NASDAQGS,1630.36,3.0,6.64,6.64,6.48,0.000000,0.025205,2026-03-02
6,3M Company (NYSE:MMM),105135,NYSE,87074.04,18.0,8.66,8.67,8.04,-0.000652,0.077737,2026-03-02
7,A. O. Smith Corporation (NYSE:AOS),4020605,NYSE,10765.87,14.0,4.02,3.80,3.80,0.058888,0.057548,2026-03-02
8,"A10 Networks, Inc. (NYSE:ATEN)",4433033,NYSE,1377.63,6.0,1.01,0.88,0.88,0.153207,0.153207,2026-03-02


In [14]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1857 entries, 4 to 1884
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   entity_name_raw  1857 non-null   object        
 1   iqid             1857 non-null   object        
 2   exchange         1857 non-null   object        
 3   mktcap_usd_m     1857 non-null   float64       
 4   n_est_fy1        1857 non-null   float64       
 5   eps_fy1_current  1857 non-null   float64       
 6   eps_fy1_1m_ago   1857 non-null   float64       
 7   eps_fy1_3m_ago   1857 non-null   float64       
 8   rev_1m           1857 non-null   float64       
 9   rev_3m           1857 non-null   float64       
 10  asof_date        1857 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(7), object(3)
memory usage: 174.1+ KB


# Store this in a database

In [15]:
import duckdb

In [16]:
con = duckdb.connect("research.duckdb")

con.execute("DROP TABLE IF EXISTS eps_revisions")

con.execute("""
CREATE TABLE eps_revisions AS
SELECT * FROM raw
""")

con.close()

In [17]:
con = duckdb.connect("research.duckdb")

result = con.execute(""" 
SELECT
    entity_name_raw,
    eps_fy1_current,
    rev_3m
FROM eps_revisions
WHERE rev_1m >= 0
LIMIT 10
""").df()

print(result)

                          entity_name_raw  eps_fy1_current    rev_3m
0  1st Source Corporation (NASDAQGS:SRCE)             6.64  0.025205
1      A. O. Smith Corporation (NYSE:AOS)             4.02  0.057548
2          A10 Networks, Inc. (NYSE:ATEN)             1.01  0.153207
3              AAON, Inc. (NASDAQGS:AAON)             1.42 -0.001406
4                    AAR Corp. (NYSE:AIR)             4.71  0.057784
5          Abbott Laboratories (NYSE:ABT)             5.68  0.101513
6                 AbbVie Inc. (NYSE:ABBV)            14.53  0.358050
7      Abercrombie & Fitch Co. (NYSE:ANF)             9.90  0.003548
8  ABM Industries Incorporated (NYSE:ABM)             3.99  0.094608
9          Acadia Realty Trust (NYSE:AKR)             0.25  0.742958
